# **CLDT - Thread - Day 10 sampai Akhir Week 2**
## **Local RTOS Accounting dan Topologi Fisik Dua Endpoint**

Panduan ini dimulai dari verdict gerbang UDP satu endpoint dan memakai header, source, build file, serta manifest yang benar - benar ada di repo. Jalur kerja menutup contract local accounting, membuktikan setiap lifecycle tanpa radio, lalu memasang endpoint kedua hanya setelah soak satu endpoint tetap lulus.

**Scope & Tags:** `Entry: Day-10 Gate` | `Thread Off For Local Pilot` | `Reconciliation First` | `Endpoint B After PASS`

---

### **Scope & Boundary Specifications**

| Domain | Specification & Minimum Requirements |
| :--- | :--- |
| **Kode yang Dibuka** | `cldt_event_trace.c`, `cldt_metrics.c`, `deadline_queue.c`, `workload.c`, dan subset local dari `endpoint_runtime.c`. |
| **Manifest Utama** | `local - rtos - baseline.jsonc` diisi dari tiga pilot. Strict JSON baru dipromosikan bila seluruh syarat `ready` benar - benar tersedia. |
| **Tetap Ditunda** | Project Thread transport, command/auth path, policy, gateway bridge, recorder host, model, power probe, dan klaim jaringan reportable. |


## **Peta Kerja Week 2**
### **Local Lifecycle Lebih Dulu, Endpoint B Setelah Soak**

Diagram ini menempatkan contract, source, build, manifest, dan hardware pada dependency yang sama. Jalur local - only sengaja tidak melewati `thread_transport.c`; radio kembali dipakai hanya pada cabang attachment endpoint B.

```mermaid
flowchart TB
    entry{"Verdict Day 10"}

    subgraph contracts["Contract local accounting"]
        types["cldt_types.h<br/>event dan terminal semantics"]
        traceh["cldt_event_trace.h<br/>ring ownership + loss policy"]
        metricsh["cldt_metrics.h<br/>lifecycle counters + reconciliation"]
        queueh["deadline_queue.h<br/>slot states + EDF ownership"]
        workloadh["workload.h<br/>stream release contract"]
        runtimeh["endpoint_runtime.h<br/>static runtime ownership"]
        types --> traceh --> metricsh
        types --> queueh --> workloadh --> runtimeh
    end

    subgraph implementation["Source dan test yang ditutup"]
        tracec["cldt_event_trace.c<br/>init, push, snapshot"] --> tracet["test_event_trace.c"]
        metricsc["cldt_metrics.c<br/>observe, audit, reconcile"] --> metricst["test_metrics.c"]
        queuec["deadline_queue.c<br/>acquire, commit, pop, release, expire"]
        workloadc["workload.c<br/>configure, start, timer, stop"]
        runtimec["endpoint_runtime.c<br/>local-only init, tasks, drain"]
        queuec --> workloadc --> runtimec
    end

    subgraph buildmanifest["Build dan manifest"]
        build["endpoint CMakeLists.txt<br/>main/CMakeLists.txt<br/>Kconfig.projbuild<br/>sdkconfig.defaults"]
        jsonc["local-rtos-baseline.jsonc<br/>pilot decisions + TODO values"]
        strict["local-rtos-baseline.json<br/>ready only after all fields resolve"]
        schema["experiment.schema.json<br/>template/ready admission"]
        jsonc --> strict --> schema
    end

    subgraph evidence["Local evidence closure"]
        pilots["critical-only<br/>bulk-only<br/>critical + bulk pilots"]
        audit["one release → one local terminal<br/>queue high-water + trace loss visible"]
        localgate{"local accounting reconciles?"}
        pilots --> audit --> localgate
    end

    entry -->|"PASS"| contracts
    entry -->|"FAIL / ambiguous"| localonly["Local path may continue<br/>network expansion stays blocked"]
    contracts --> implementation --> build --> pilots
    build --> jsonc
    localonly --> contracts
    localgate -->|"PASS and one-endpoint soak remains stable"| endpointb["Flash upstream ot_cli on endpoint B<br/>attach, record role/parent/partition"]
    localgate -->|"FAIL"| repair["Keep Thread off<br/>repair lifecycle/accounting"]
    endpointb --> week3["Week 3 entry<br/>project frames + bounded evidence path"]

    classDef contract fill:#312e81,stroke:#a78bfa,color:#f5f3ff;
    classDef code fill:#0c4a6e,stroke:#38bdf8,color:#e0f2fe;
    classDef evidence fill:#14532d,stroke:#22c55e,color:#dcfce7;
    classDef decision fill:#713f12,stroke:#fbbf24,color:#fef3c7;
    class types,traceh,metricsh,queueh,workloadh,runtimeh contract;
    class tracec,tracet,metricsc,metricst,queuec,workloadc,runtimec,build code;
    class jsonc,strict,schema,pilots,audit,endpointb,week3 evidence;
    class entry,localgate,localonly,repair decision;
```


## **Step 01: Membawa Verdict Day 10 Apa Adanya**
**Track:** **`Entry Gate`**

Week 2 tidak dimulai dari “UDP pernah muncul di console”. Input yang dipakai adalah satu block cold boot dan soak yang sudah memiliki parameter tetap, hash image, raw log, serta verdict. Semua nilai pada kolom contoh di bawah hanyalah bentuk isian dan diganti dengan hasil lembar 1-10. Bila verdict masih gagal atau ambigu, pekerjaan local accounting boleh berjalan pada meja terpisah, tetapi endpoint kedua dan perluasan jaringan belum masuk.

### **Data yang dibawa dari lembar sebelumnya**

| Catatan dari gate 1-10 | Contoh format - ganti dengan hasil aktual |
| - | - |
| Verdict gate | e.g. PASS |
| Alasan verdict | e.g. PASS - 3/3 cold boot memenuhi threshold yang dibekukan |
| Commit ESP - IDF | e.g. 40 karakter hexadecimal dari checkout yang dibangun |
| SHA - 256 binary RCP | e.g. 64 karakter hexadecimal |
| SHA - 256 binary gateway upstream | e.g. 64 karakter hexadecimal |
| SHA - 256 binary endpoint A upstream | e.g. 64 karakter hexadecimal |
| `setup.thread_channel` | e.g. 15 |
| Identitas/hash active dataset | e.g. SHA - 256 dari dataset export yang disimpan privat |
| Repetisi cold boot valid | e.g. 3 |
| Aggregate UDP delivery ratio | e.g. 0.9989 |
| Total reset tak terduga | e.g. 0 |
| Total detach | e.g. 0 |
| Lokasi raw log | e.g. C:\logs\cldt\day - 10 - gate |

### **Keputusan masuk**

1. Hasil gerbang PASS membuka dua pekerjaan: local RTOS accounting dan persiapan endpoint B.
2. Local accounting memakai satu C6 dengan firmware project dan radio tidak diinisialisasi.
3. Endpoint B tidak bergabung ke Thread sebelum block satu - endpoint selesai.
4. Kegagalan local accounting menahan Week 3. Recorder atau model tidak boleh dipakai untuk menyamarkan lifecycle item yang belum dapat direkonsiliasi.
5. Image upstream Day 10 tidak ditimpa tanpa menyimpan binary hash dan `sdkconfig`; endpoint A dapat di - flash ulang untuk local pilot, lalu dikembalikan ke image upstream yang identitasnya sama saat topologi dua endpoint diuji.



> **Entry rule:** bila kolom verdict belum PASS, bagian attachment endpoint B tetap kosong. Tidak ada “conditional pass” yang diciptakan setelah melihat hasil.


## **Step 02: Lima Keputusan yang Harus Jelas Sebelum Stub Diisi**
**Track:** **`Contract Before Code`**

Beberapa komentar TODO meminta perilaku yang belum seluruhnya dapat direpresentasikan oleh struktur saat ini. Keputusan berikut ditulis di catatan implementasi lebih dahulu; setelah itu barulah fungsi diisi.

### **1. Jalur local - only**

`sdkconfig.defaults` mengaktifkan komponen OpenThread, tetapi radio tetap tidak aktif selama tidak ada fungsi init/attach yang dipanggil. Build lokal membutuhkan mekanisme yang dapat direproduksi untuk membuat supervisor hanya memulai queue, workload, trace, dan accounting. Repository belum menyediakan switch Kconfig khusus local - only.

Pilihan yang dicatat:

| Keputusan implementasi | Contoh bentuk jawaban - ganti setelah source ditetapkan |
| - | - |
| Cara menjalankan jalur local - only | e.g. cabang di `cldt_endpoint_runtime_start()` tidak memanggil `cldt_thread_transport_init()` |
| Bukti OpenThread tidak diinisialisasi | e.g. call path source dan cold - boot log tidak pernah memasuki attachment |
| Bukti Wi - Fi tidak diinisialisasi | e.g. call path source dan cold - boot log tanpa station start |
| Identitas build lokal | e.g. source commit, SHA - 256 `sdkconfig`, dan SHA - 256 binary |

Build option, commit, dan log cold boot masuk ke `firmware_reference` manifest. Absennya packet pada satu pengamatan bukan satu - satunya bukti; call path yang tidak pernah menginisialisasi radio juga harus dapat ditunjukkan.

### **2. Static allocation**

Komentar `workload.c` dan `endpoint_runtime.c` meminta task, timer, queue, event group, serta trace storage dialokasikan statis. Struct sekarang hanya menyimpan handle; ia belum memiliki `StaticTask_t`, stack array, `StaticTimer_t`, `StaticEventGroup_t`, atau array `cldt_trace_record_t`.

Sebelum mengisi TODO, pilih apakah storage:

1. dimiliki oleh `cldt_endpoint_runtime_t`; atau
2. diberikan caller melalui config/storage struct.

Campuran heap dan static allocation tidak dianggap memenuhi kontrak. Ukuran stack dan trace capacity dicatat dari high - water/overflow evidence, bukan dari tebakan yang kemudian dibesarkan diam - diam.

### **3. Reservation dan full - queue admission**

`cldt_deadline_queue_acquire()` tidak menerima traffic class atau deadline. Namun komentarnya meminta reserved capacity, sementara `cldt_deadline_queue_commit()` meminta membandingkan incoming deadline dengan item paling lambat saat queue penuh. Bila semua slot sudah queued, incoming item bahkan belum memiliki slot.

Sebelum implementasi, kontrak API harus memilih salah satu:

- class/deadline diketahui saat acquire; atau
- ada producer scratch slot terpisah dari queue capacity; atau
- admission dilakukan sebelum acquire melalui fungsi terpisah.

Nilai yang dibekukan:

| Kontrak queue yang perlu ditutup di source | Contoh status/catatan |
| - | - |
| Jumlah slot yang dicadangkan untuk `CLDT_TRAFFIC_CONTROL` dan `CLDT_TRAFFIC_CRITICAL` | e.g. 4; nilai final harus berada dalam `capacity` dan dibuktikan pilot |
| Cara `cldt_deadline_queue_acquire()` mengetahui `traffic_class` | e.g. NOT RESOLVED - signature saat ini belum menerima class |
| Cara incoming item dibandingkan ketika semua slot sudah queued | e.g. NOT RESOLVED - contract saat ini belum menyediakan incoming slot |
| Identity untuk coalescing `CLDT_TRAFFIC_TELEMETRY` | e.g. `meta.node_id` ditambah stream identity yang benar - benar tersedia pada payload |


### **4. Satu timer untuk beberapa stream**

`cldt_workload_t` memiliki satu `release_timer` tetapi dapat memuat delapan stream. Jalur paling sederhana adalah satu timer untuk next absolute release: producer mencari waktu release terdekat, memproses semua stream yang jatuh pada waktu itu, kemudian menjadwalkan absolute intent berikutnya. Relative chaining yang menumpuk drift tidak dipakai.

### **5. Terminal lokal**

Eksperimen ini tanpa network. `CLDT_EVENT_MESSAGE_ACK` tidak boleh dipalsukan sebagai hasil lokal. Sebelum reducer metrics ditulis, tentukan bagaimana `CLDT_EVENT_TASK_FINISH` direpresentasikan pada aggregate. Konservasi lokal memerlukan satu terminal untuk setiap `CLDT_EVENT_TASK_RELEASE`: `CLDT_EVENT_TASK_FINISH`, `CLDT_EVENT_MESSAGE_EXPIRE`, `CLDT_EVENT_MESSAGE_COALESCE`, `CLDT_EVENT_QUEUE_REJECT`, `CLDT_EVENT_MESSAGE_DROP`, atau item yang memang masih `unresolved` pada snapshot.

Ada blocker nyata pada contract saat ini: `cldt_metrics_t` tidak mempunyai counter yang secara eksplisit mewakili `CLDT_EVENT_TASK_FINISH`. Notebook tidak memberi nama field baru. Arti terminal itu harus diperbaiki di header/source dan test sebelum `cldt_metrics_reconcile()` dapat dipakai untuk klaim local - only; memakai `acknowledged` untuk local finish akan mencampur completion CPU dengan delivery jaringan.

| Keputusan contract | Contoh status/catatan |
| - | - |
| Terminal untuk local - only | e.g. `CLDT_EVENT_TASK_FINISH` |
| Representasi terminal tersebut di `cldt_metrics_t` | e.g. NOT RESOLVED - jangan memakai `acknowledged` sebagai pengganti |
| Definisi selesai tepat waktu | e.g. `local_time_us <= deadline_local_us` pada `CLDT_EVENT_TASK_FINISH` |
| Item tanpa terminal pada snapshot | e.g. masuk `cldt_reconciliation_t.unresolved`, bukan dihapus |


## **Header yang Dipakai Reducer Week 2**
### **`cldt_types.h` sebagai Vocabulary Lifecycle**

`cldt_event_trace.c`, `cldt_metrics.c`, `deadline_queue.c`, `workload.c`, dan `endpoint_runtime.c` semuanya memakai tipe pada header ini. Snippet berikut disalin persis dari `common/include/cldt/cldt_types.h`; nama enum, struct, dan field di dalamnya adalah identifier repo, bukan label notebook.

~~~c
#ifndef CLDT_TYPES_H
#define CLDT_TYPES_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_PROTOCOL_MAGIC UINT16_C(0x434C)
#define CLDT_PROTOCOL_VERSION UINT8_C(1)
#define CLDT_WIRE_HEADER_BYTES UINT16_C(72)
#define CLDT_MAX_PAYLOAD_BYTES UINT16_C(256)
#define CLDT_AUTH_TAG_BYTES 16U
#define CLDT_TRACE_DETAIL_BYTES 24U
#define CLDT_POLICY_STREAM_COUNT 4U
#define CLDT_COMMAND_AUTHORITY_NODE_ID UINT32_C(0)

typedef uint32_t cldt_node_id_t;
typedef uint64_t cldt_run_id_t;
typedef uint32_t cldt_boot_id_t;
typedef uint32_t cldt_sequence_t;
typedef uint32_t cldt_policy_epoch_t;

typedef enum {
    CLDT_NODE_GATEWAY_HOST = 0,
    CLDT_NODE_RADIO_COPROCESSOR,
    CLDT_NODE_ROUTER_ENDPOINT,
    CLDT_NODE_LOW_POWER_ENDPOINT
} cldt_node_role_t;

typedef enum {
    CLDT_TRAFFIC_CONTROL = 0,
    CLDT_TRAFFIC_CRITICAL,
    CLDT_TRAFFIC_TELEMETRY,
    CLDT_TRAFFIC_BULK,
    CLDT_TRAFFIC_COUNT
} cldt_traffic_class_t;

typedef enum {
    CLDT_FRAME_OBSERVATION = 0,
    CLDT_FRAME_COMMAND,
    CLDT_FRAME_ACKNOWLEDGEMENT,
    CLDT_FRAME_CLOCK_SYNC,
    CLDT_FRAME_HEALTH
} cldt_frame_kind_t;

typedef enum {
    CLDT_EVENT_TASK_RELEASE = 0,
    CLDT_EVENT_TASK_START,
    CLDT_EVENT_TASK_FINISH,
    CLDT_EVENT_TASK_BLOCK,
    CLDT_EVENT_QUEUE_ENQUEUE,
    CLDT_EVENT_QUEUE_DEQUEUE,
    CLDT_EVENT_QUEUE_REJECT,
    CLDT_EVENT_POOL_EXHAUSTION,
    CLDT_EVENT_MESSAGE_SEND,
    CLDT_EVENT_MESSAGE_ACK,
    CLDT_EVENT_MESSAGE_EXPIRE,
    CLDT_EVENT_MESSAGE_COALESCE,
    CLDT_EVENT_MESSAGE_DROP,
    CLDT_EVENT_MESSAGE_DUPLICATE,
    CLDT_EVENT_LINK_CHANGE,
    CLDT_EVENT_POWER_SAMPLE,
    CLDT_EVENT_POLICY_APPLY,
    CLDT_EVENT_POLICY_REJECT,
    CLDT_EVENT_POLICY_FALLBACK,
    CLDT_EVENT_HEALTH,
    CLDT_EVENT_COUNT
} cldt_event_kind_t;

typedef enum {
    CLDT_GATE_COLD = 0,
    CLDT_GATE_OBSERVE,
    CLDT_GATE_TRUSTED,
    CLDT_GATE_ABSTAIN
} cldt_gate_state_t;

typedef enum {
    CLDT_MODEL_NAIVE = 0,
    CLDT_MODEL_NETWORK_ONLY,
    CLDT_MODEL_CROSS_LAYER,
    CLDT_MODEL_VARIANT_COUNT
} cldt_model_variant_t;

/*
 * In-memory metadata. It is not a packed wire structure. Encoding and decoding
 * must be performed field by field through cldt_protocol.h.
 *
 * Identity is frame-kind specific. For observations, acknowledgements, health,
 * and trace-bearing frames, node_id/boot_id identify the emitting device. A
 * version 1 command is one global policy datagram for every endpoint admitted
 * to the run: node_id is CLDT_COMMAND_AUTHORITY_NODE_ID and boot_id identifies
 * the host coordinator process that issued it, not a destination. The gateway
 * guards and forwards those identical bytes. Version 1 does not define
 * different authenticated command bytes per endpoint.
 */
typedef struct {
    cldt_frame_kind_t kind;
    cldt_traffic_class_t traffic_class;
    uint16_t flags;
    uint8_t hop_limit;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t transmit_local_us;
    uint64_t deadline_local_us;
} cldt_frame_meta_t;

/*
 * Decoder output borrows payload memory from the input byte buffer. The caller
 * must keep that buffer alive and unchanged while this view is in use.
 */
typedef struct {
    cldt_frame_meta_t meta;
    const uint8_t *payload;
    uint16_t payload_bytes;
    uint32_t crc32c;
    uint8_t authentication_tag[CLDT_AUTH_TAG_BYTES];
} cldt_frame_view_t;

typedef struct {
    cldt_event_kind_t kind;
    /* Every work-item event carries its class; HEALTH may use CLDT_TRAFFIC_COUNT. */
    cldt_traffic_class_t traffic_class;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t local_time_us;
    /*
     * Work-item events repeat the item's release and absolute deadline in the
     * same local monotonic clock domain as local_time_us. Non-work-item events
     * store zero in both fields. This permits stateless aggregate timing while
     * preserving raw timestamps for a separate per-item lifecycle audit.
     */
    uint64_t release_local_us;
    uint64_t deadline_local_us;
    uint32_t task_id;
    int8_t core_id;
    uint16_t queue_depth;
    int16_t link_rssi_dbm;
    uint32_t time_uncertainty_us;
    /* Fixed-size auxiliary bytes; each event kind documents its own encoding. */
    uint8_t detail[CLDT_TRACE_DETAIL_BYTES];
} cldt_trace_record_t;

typedef struct {
    uint32_t release_period_ms[CLDT_POLICY_STREAM_COUNT];
    uint32_t phase_offset_ms[CLDT_POLICY_STREAM_COUNT];
    uint16_t burst_limit[CLDT_POLICY_STREAM_COUNT];
    uint16_t batch_size[CLDT_POLICY_STREAM_COUNT];
    uint32_t token_rate_milli_pps[CLDT_POLICY_STREAM_COUNT];
    cldt_policy_epoch_t epoch;
    uint64_t issued_gateway_us;
    uint32_t ttl_ms;
} cldt_policy_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t horizon_start_host_us;
    uint64_t horizon_end_host_us;
    uint64_t evaluated_host_us;
    uint64_t newest_observation_host_us;
    uint32_t sample_count;
    uint32_t model_lag_us;
    uint32_t clock_uncertainty_us;
    double relative_p95_error;
    double pdr_error_points;
    double prediction_interval_coverage;
    /* False when required horizon evidence is missing, stale, or unreconciled. */
    bool observation_integrity_valid;
    bool inside_calibrated_region;
} cldt_fidelity_sample_t;

#ifdef __cplusplus
}
#endif

#endif
~~~

Pekerjaan header pada Week 2 dibatasi pada contract yang dibutuhkan local accounting:

1. Makna `cldt_trace_record_t.detail` dibekukan untuk event release, queue admission/rejection, finish, expiry, coalesce, dan drop yang benar - benar akan direkam.
2. `CLDT_EVENT_TASK_FINISH` diberi representasi aggregate yang tidak disamakan dengan `CLDT_EVENT_MESSAGE_ACK`. Jika contract itu memerlukan perubahan `cldt_metrics_t`, perubahan dilakukan di header tersebut bersama `cldt_metrics.c` dan `test_metrics.c`; notebook tidak menciptakan nama field pengganti.
3. `run_id`, `node_id`, `boot_id`, dan `sequence` tetap menjadi identity satu logical item pada seluruh trace dan audit.
4. Nilai not - applicable untuk metadata jaringan pada pilot local - only ditulis normatif. Nilai lama dari run jaringan tidak boleh terbawa seolah - olah merupakan pengamatan lokal.
5. Policy, fidelity sample, command authority, dan remote actuation yang juga tercantum pada header tidak diimplementasikan pada Week 2.


## **Step 03: Mengisi Ring Buffer Tanpa Menghapus Bukti Lama**
**Track:** **`Common - Local Trace`**

File yang dikerjakan adalah `common/src/cldt_event_trace.c`. Ia tidak mengalokasikan memory dan tidak membuat lock; caller menyediakan storage serta concurrency model. Untuk local pilot, model paling mudah dipertanggungjawabkan adalah satu producer owner dan satu trace consumer, atau satu short critical section yang sama untuk semua producer.

### **Header contract: `common/include/cldt/cldt_event_trace.h`**

~~~c
#ifndef CLDT_EVENT_TRACE_H
#define CLDT_EVENT_TRACE_H

#include <stddef.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    cldt_trace_record_t *records;
    size_t capacity;
    size_t read_index;
    size_t write_index;
    size_t count;
    uint64_t dropped_records;
} cldt_event_trace_t;

/* Uses caller-owned storage. Concurrency protection belongs to the adapter. */
cldt_status_t cldt_event_trace_init(
    cldt_event_trace_t *trace,
    cldt_trace_record_t *storage,
    size_t capacity);

cldt_status_t cldt_event_trace_push(
    cldt_event_trace_t *trace,
    const cldt_trace_record_t *record);

cldt_status_t cldt_event_trace_pop(
    cldt_event_trace_t *trace,
    cldt_trace_record_t *record);

#ifdef __cplusplus
}
#endif

#endif

~~~

Header ini menentukan caller - owned `records`, `capacity`, index, `count`, dan `dropped_records`. Implementasi source tidak boleh membuat field bayangan di notebook atau adapter.

### **Source TODO asli**

~~~c
#include "cldt/cldt_event_trace.h"

cldt_status_t cldt_event_trace_init(
    cldt_event_trace_t *trace,
    cldt_trace_record_t *storage,
    size_t capacity)
{
    (void)trace;
    (void)storage;
    (void)capacity;

    /*
     * IMPLEMENTATION TODO: reject null trace/storage and zero capacity; bind
     * only caller-owned storage; set both indices and count to zero; and set
     * dropped_records to zero. This object performs no allocation and no lock
     * creation. Its caller must select the single-producer/single-consumer or
     * externally synchronized usage model before calling init.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_event_trace_push(
    cldt_event_trace_t *trace,
    const cldt_trace_record_t *record)
{
    (void)trace;
    (void)record;

    /*
     * IMPLEMENTATION TODO: choose one loss policy before implementation. For
     * this research trace, reject the incoming record, increment dropped_records,
     * and return a visible overflow status; do not overwrite old evidence.
     * Reject kind >= CLDT_EVENT_COUNT. Work-item events require a real traffic
     * class plus internally ordered release/deadline timestamps; link, power,
     * policy, and health events use documented zero/not-applicable values.
     * Copy one complete record only after capacity is established, advance the
     * write index modulo capacity, and update count atomically for the selected
     * concurrency model. Test wraparound, full buffer, invalid kind, and null
     * record paths.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_event_trace_pop(
    cldt_event_trace_t *trace,
    cldt_trace_record_t *record)
{
    (void)trace;
    (void)record;

    /*
     * IMPLEMENTATION TODO: reject null arguments, return CLDT_ERR_NOT_READY when
     * count is zero, copy exactly one record into caller storage, then advance
     * read_index modulo capacity and decrement count. Never return a pointer to
     * ring storage because an immediate producer write could overwrite it. Test
     * FIFO order across wraparound and confirm pop never changes dropped_records.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}
~~~

### **Urutan pengisian**

1. `cldt_event_trace_init()` menolak pointer null dan capacity nol, mengikat storage milik caller, lalu membuat `read_index`, `write_index`, `count`, dan `dropped_records` bernilai nol.
2. Object yang belum di - init tidak dipakai. Invariant dasarnya: `count <= capacity` dan kedua index selalu berada pada rentang `[0, capacity)`.
3. `cldt_event_trace_push()` memvalidasi kind serta timestamp/class sesuai jenis record sebelum menulis. Saat penuh, incoming record ditolak, `dropped_records` bertambah, record lama tetap utuh.
4. Write dilakukan pada `records[write_index]`, index bergerak modulo capacity, kemudian count berubah menurut concurrency rule yang dipilih.
5. `cldt_event_trace_pop()` pada buffer kosong mengembalikan `CLDT_ERR_NOT_READY`. Pada kondisi tersedia, satu record disalin keluar sebelum read index dan count diperbarui.
6. Pointer ke internal ring tidak pernah dikembalikan; consumer selalu menerima copy.
7. Overflow menghasilkan health/error evidence. Capacity tidak dibesarkan sampai drop “menghilang” tanpa catatan.

### **Test file yang sudah ada**

~~~c
#include <stdio.h>

#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_metrics.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Build deterministic traces for release, admission, dequeue, ACK,
     *    expiry, rejection, coalescing, drop, restart, and fallback events.
     * 2. Replay one injected fault at a time: truncation, corruption,
     *    duplication, reordering, missing terminal, stale observation, wrong
     *    run, boot-ID change, and durable replay-state loss.
     * 3. Assert item-level audit and aggregate reconciliation independently;
     *    layer-specific MAC attempt/ACK diagnostics are not forced into false
     *    equality with application messages.
     * 4. Assert missing or unreconciled evidence produces an explicit invalid
     *    observation input and can never become a favorable gate sample.
     */
    fprintf(stderr, "SKIP: event-trace and deterministic replay tests have not been implemented.\n");
    return 77;
}
~~~

Checklist test tersebut memuat fault Week 5, sehingga `return 77` belum boleh diubah hanya karena ring buffer dasar selesai. Pada Week 2, test yang ditambahkan lebih dahulu adalah null input, zero capacity, FIFO, wraparound, full - buffer rejection, `dropped_records`, invalid kind, dan pop - empty. Bagian restart, fallback, wrong - run, serta replay - state loss tetap belum selesai dan target keseluruhan tetap skip.

### **Catatan hasil**

| Field/pemeriksaan | Contoh format - ganti dengan hasil aktual |
| - | - |
| `cldt_event_trace_t.capacity` | e.g. 64 records |
| Concurrency model milik adapter | e.g. satu producer dan satu consumer, atau satu critical section pendek |
| FIFO setelah wraparound | e.g. PASS - urutan record tetap benar |
| Push saat penuh | e.g. PASS - `CLDT_ERR_NO_SPACE` dan `dropped_records == 1` |
| Kind/pointer tidak valid | e.g. PASS - status yang diharapkan cocok |
| Puncak `cldt_event_trace_t.count` saat pilot | e.g. 37 |
| `cldt_event_trace_t.dropped_records` setelah pilot | e.g. 0 |


## **Step 04: Reducer, Aggregate Reconciliation, dan Item Audit**
**Track:** **`Common - Accounting`**

File `common/src/cldt_metrics.c` adalah pusat klaim Week 2. Aggregate counter saja tidak cukup; lifecycle juga harus diperiksa per identity `(run_id, node_id, boot_id, sequence)`. Keputusan terminal lokal dari bagian 02 harus sudah selesai sebelum switch event ditulis.

### **Header contract: `common/include/cldt/cldt_metrics.h`**

~~~c
#ifndef CLDT_METRICS_H
#define CLDT_METRICS_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * All counters refer to unique logical work items identified by run, node,
 * boot, and sequence. A transport retry is trace detail, not another sent
 * item. Aggregate counters are useful only after the item-identity audit below;
 * equal totals alone cannot prove that one item was not counted twice while
 * another item disappeared.
 */
typedef struct {
    uint64_t released[CLDT_TRAFFIC_COUNT];
    uint64_t admitted[CLDT_TRAFFIC_COUNT];
    uint64_t sent[CLDT_TRAFFIC_COUNT];
    uint64_t acknowledged[CLDT_TRAFFIC_COUNT];
    uint64_t on_time_acknowledged[CLDT_TRAFFIC_COUNT];
    uint64_t deadline_missed[CLDT_TRAFFIC_COUNT];
    uint64_t expired[CLDT_TRAFFIC_COUNT];
    uint64_t coalesced[CLDT_TRAFFIC_COUNT];
    uint64_t rejected[CLDT_TRAFFIC_COUNT];
    uint64_t dropped[CLDT_TRAFFIC_COUNT];
    uint64_t duplicated[CLDT_TRAFFIC_COUNT];
    uint64_t response_time_sum_us[CLDT_TRAFFIC_COUNT];
    uint64_t lateness_sum_us[CLDT_TRAFFIC_COUNT];
    uint32_t queue_high_water;
    uint32_t pool_exhaustions;
    uint64_t measurement_duration_us;
    uint64_t energy_uj;
} cldt_metrics_t;

/*
 * Reconciliation is a computed report, not a mutable metric. A class is
 * consistent only when every released work item is terminal or explicitly
 * unresolved at the snapshot boundary.
 */
typedef struct {
    uint64_t terminal[CLDT_TRAFFIC_COUNT];
    uint64_t unresolved[CLDT_TRAFFIC_COUNT];
    bool consistent[CLDT_TRAFFIC_COUNT];
} cldt_reconciliation_t;

/*
 * Result of auditing raw work-item lifecycles by full logical identity. This
 * is deliberately separate from aggregate reconciliation so reports cannot
 * mistake balanced counter corruption for complete evidence.
 */
typedef struct {
    uint64_t logical_items;
    uint64_t duplicate_releases;
    uint64_t duplicate_terminals;
    uint64_t terminal_without_release;
    uint64_t unresolved_items;
    bool consistent;
} cldt_item_audit_t;

void cldt_metrics_reset(cldt_metrics_t *metrics);

cldt_status_t cldt_metrics_record_trace(
    cldt_metrics_t *metrics,
    const cldt_trace_record_t *record);

/*
 * Calculates per-class aggregate conservation without modifying metrics. This
 * is a necessary check, not proof of per-item uniqueness. The caller must also
 * require a consistent cldt_item_audit_t and archive unresolved work rather
 * than discarding it before computing rates.
 */
cldt_status_t cldt_metrics_reconcile(
    const cldt_metrics_t *metrics,
    cldt_reconciliation_t *output);

/*
 * Audits work-item lifecycles in a trace sorted lexicographically by run_id,
 * node_id, boot_id, sequence, and local_time_us. The function skips non-item
 * events and never reorders caller-owned storage. Sorting belongs to the host
 * analysis/recorder boundary because embedded targets must not allocate an
 * unbounded identity table. A reportable run requires this audit and aggregate
 * reconciliation to pass.
 */
cldt_status_t cldt_metrics_audit_sorted_trace(
    const cldt_trace_record_t *records,
    size_t record_count,
    cldt_item_audit_t *output);

#ifdef __cplusplus
}
#endif

#endif

~~~

Semua counter, output reconciliation, dan item - audit field yang dipakai pada cell ini berasal dari header tersebut. Gap terminal local juga terlihat langsung karena `cldt_metrics_t` belum mempunyai field khusus untuk `CLDT_EVENT_TASK_FINISH`.

### **Source TODO asli**

~~~c
#include "cldt/cldt_metrics.h"

void cldt_metrics_reset(cldt_metrics_t *metrics)
{
    (void)metrics;

    /*
     * IMPLEMENTATION TODO: validate metrics is non-null, clear every counter
     * and high-water value in one operation, and call this only after the
     * supervisor has acknowledged a run boundary. Do not reuse it to hide a
     * mid-run accounting problem. A test should prove that reset creates an
     * all-zero snapshot and does not retain a previous traffic-class value.
     */
}

cldt_status_t cldt_metrics_record_trace(
    cldt_metrics_t *metrics,
    const cldt_trace_record_t *record)
{
    (void)metrics;
    (void)record;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject null arguments, kind >= CLDT_EVENT_COUNT, and an invalid traffic
     *    class for every work-item event.
     * 2. Map release, admission, send, acknowledgement, expiry, coalescing,
     *    rejection, drop, duplicate, and pool-exhaustion events to exactly one
     *    documented field. Coalescing and exhaustion have dedicated event kinds;
     *    do not infer them from a generic rejection detail byte.
     * 3. Update queue high-water and pool exhaustion only from their respective
     *    authoritative records; do not infer either from packet loss.
     * 4. Count response time and lateness only on the authoritative terminal
     *    event when release_local_us, deadline_local_us, and local_time_us are
     *    ordered in the same monotonic domain. Use checked subtraction.
     * Keep this mapping as a table or a clearly exhaustive switch and write a
     * unit test for every event kind. This reducer does not remember item IDs;
     * cldt_metrics_audit_sorted_trace() is the separate uniqueness proof. A
     * silent default case is not acceptable.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_metrics_reconcile(
    const cldt_metrics_t *metrics,
    cldt_reconciliation_t *output)
{
    (void)metrics;
    (void)output;

    /*
     * IMPLEMENTATION TODO: for each traffic class, calculate whether released
     * work equals all terminal outcomes plus explicitly unresolved in-flight
     * work. Detect aggregate impossibilities such as acknowledgements above
     * sends or expired items above admissions. Do not claim this aggregate
     * function can identify a duplicated terminal for item A balanced by a
     * missing terminal for item B; the identity audit handles that case. Return
     * a reconciliation error with no mutation. The report layer must require
     * both checks before turning counters into a rate.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_metrics_audit_sorted_trace(
    const cldt_trace_record_t *records,
    size_t record_count,
    cldt_item_audit_t *output)
{
    (void)records;
    (void)record_count;
    (void)output;

    /*
     * IMPLEMENTATION TODO:
     * 1. Accept a null records pointer only when record_count is zero; require
     *    output and clear a local result before examining caller-owned data.
     * 2. Skip link, power, policy, and health records. For work-item records,
     *    validate kind/class/timestamps and require nondecreasing lexical order
     *    by (run_id, node_id, boot_id, sequence, local_time_us).
     * 3. Scan one identity group at a time. Require exactly one release, at most
     *    one terminal outcome, no terminal without release, and no event after a
     *    terminal except an explicitly documented transport-duplicate detail.
     * 4. Count an item with a release but no terminal as unresolved rather than
     *    deleting it. Set consistent only when duplicate_releases,
     *    duplicate_terminals, terminal_without_release, and unresolved_items
     *    are all zero. Publish output only after the full scan succeeds.
     * Add tests where aggregate totals balance despite one duplicated terminal
     * and one missing terminal; this audit must reject that trace.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}
~~~

### **Cara mengerjakannya**

1. `cldt_metrics_reset()` membersihkan seluruh struct pada run boundary yang telah diakui supervisor. Reset di tengah run dilarang karena dapat membuat loss hilang dari laporan.
2. `cldt_metrics_record_trace()` menggunakan exhaustive switch. Setiap event kind memiliki tepat satu arti; policy/link/health event tidak menaikkan delivery counter.
3. Penjumlahan counter dan timing memakai checked arithmetic. Overflow tidak wrap menjadi angka kecil; status error atau saturation policy dicatat dan diuji.
4. Response time dan lateness hanya dihitung dari terminal yang authoritative dan timestamp satu clock domain dengan urutan yang valid.
5. `cldt_metrics_reconcile()` memeriksa conservation per traffic class serta impossibility seperti terminal lebih banyak daripada release/admission.
6. `unresolved` hanya boleh mewakili item yang memang masih in - flight pada snapshot boundary. Ia tidak boleh menjadi keranjang untuk event yang hilang.
7. `cldt_metrics_audit_sorted_trace()` menerima trace yang sudah diurutkan oleh caller. Fungsi tidak mengurutkan dan tidak mengalokasikan identity table.
8. Setiap identity group memerlukan satu release dan maksimal satu terminal. Duplicate release, duplicate terminal, terminal tanpa release, serta unresolved semuanya terlihat terpisah.
9. `consistent` hanya true bila keempat counter inconsistency nol.
10. PDR, on - time ratio, atau rata - rata timing baru dihitung setelah aggregate reconciliation dan item audit sama - sama lulus.

### **Mapping yang dibekukan**

| Event | Counter/arti | Terminal? |
| - | - | - |
| `CLDT_EVENT_TASK_RELEASE` | `released` | tidak |
| `CLDT_EVENT_QUEUE_ENQUEUE` | `admitted` | tidak |
| `CLDT_EVENT_TASK_FINISH` | contract aggregate belum terwakili oleh field khusus di `cldt_metrics_t` | terminal local pada item audit |
| `CLDT_EVENT_MESSAGE_EXPIRE` | `expired` | ya |
| `CLDT_EVENT_MESSAGE_COALESCE` | `coalesced` | ya untuk item yang digantikan |
| `CLDT_EVENT_QUEUE_REJECT` | `rejected` | ya |
| `CLDT_EVENT_MESSAGE_DROP` | `dropped` | ya |
| `CLDT_EVENT_MESSAGE_DUPLICATE` | `duplicated` | bukan logical release baru |
| `CLDT_EVENT_POOL_EXHAUSTION` | `pool_exhaustions` | health/admission evidence |

Tabel tersebut belum menentukan mapping aggregate `CLDT_EVENT_TASK_FINISH`; nilai final mengikuti keputusan bagian 02 dan perubahan contract yang dibuat di source/header.


## **Step 05: Membuat Accounting Test Benar - Benar Keluar dari Skip**
**Track:** **`Host Test - Metrics`**

Berbeda dari `test_event_trace` yang masih membawa kasus fase lanjut, checklist `tests/test_metrics.c` langsung mendukung accounting gate. File ini menjadi target host yang diselesaikan pada Week 2.

### **Test TODO asli**

~~~c
#include <stdio.h>

#include "cldt/cldt_metrics.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Construct trace records carrying traffic class, run ID, node ID, boot
     *    ID, sequence, release/deadline/current timestamps, and terminal event.
     *    Assert the expected counter and checked timing delta after every record;
     *    test every kind through CLDT_EVENT_COUNT, including explicit coalescing
     *    and pool exhaustion plus policy events that must not inflate delivery.
     * 2. Build conservation cases for acknowledged, expired, coalesced, rejected,
     *    dropped, duplicated, and genuinely unresolved work. Verify the report
     *    distinguishes an incomplete run from a mathematically inconsistent one.
     * 3. Sort raw records by the documented full identity and test the item audit
     *    with acknowledgement before release, two terminal outcomes for one item,
     *    a terminal without release, unresolved work, invalid class/kind, and
     *    out-of-order input. Include the counterbalanced case where item A has two
     *    terminals and item B has none: aggregate totals may balance, but the item
     *    audit must fail.
     * 4. Add counter saturation and reset-at-run-boundary cases. Do not calculate
     *    PDR, deadline ratio, or energy efficiency until both aggregate
     *    reconciliation and per-item audit succeed.
     */
    fprintf(stderr, "SKIP: metric-accounting tests have not been implemented.\n");
    return 77;
}
~~~

### **Urutan case**

1. Satu record release dibuat dengan seluruh identity dan timestamp eksplisit.
2. Admission serta terminal lokal ditambahkan satu per satu; counter diperiksa setelah setiap call, bukan hanya pada akhir.
3. Jalur terminal terpisah dibuat untuk expire, coalesce, reject, dan drop.
4. Event policy/link/health masuk reducer tetapi tidak boleh menaikkan release atau terminal work - item.
5. Aggregate case mencakup complete, valid - but - unresolved - at - snapshot, dan mathematically impossible.
6. Item audit mencakup input sorted yang valid, acknowledgement/terminal sebelum release, duplicate release, dua terminal, terminal tanpa release, unresolved, invalid class/kind, dan input out - of - order.
7. Counterbalanced corruption wajib ada: item A mempunyai dua terminal dan item B tidak mempunyai terminal. Aggregate dapat tampak seimbang, item audit harus gagal.
8. Reset hanya diuji pada run boundary. Saturation/overflow policy diuji sesuai keputusan implementation.
9. `return 77` diganti dengan exit sukses hanya setelah seluruh checklist source test - termasuk edge case - ada dan pass.

~~~powershell
cmake -S . -B build -DCLDT_BUILD_TESTS=ON
cmake --build build --parallel
ctest --test-dir build -R "^test_metrics$" --output-on-failure
~~~

### **Hasil test**

| Pemeriksaan `test_metrics` | Contoh format - ganti dengan hasil aktual |
| - | - |
| Exit code | e.g. 0 setelah seluruh checklist TODO selesai |
| Event kind yang diuji | e.g. seluruh nilai dari 0 sampai sebelum `CLDT_EVENT_COUNT` |
| Complete lifecycle | e.g. PASS |
| Valid unresolved snapshot | e.g. PASS - jumlah `unresolved` cocok |
| Aggregate yang mustahil | e.g. PASS - fungsi menolak |
| Dua terminal pada item A dan nol pada item B | e.g. PASS - aggregate dapat seimbang, item audit tetap menolak |
| Reset pada run boundary | e.g. PASS - seluruh counter kembali nol |
| Compiler dan lokasi log | e.g. gcc 14.2.0; C:\logs\cldt\test - metrics.txt |


## **Step 06: Menjaga Ownership Slot dan Deadline Order**
**Track:** **`Endpoint - Fixed Pool - EDF`**

`cldt_deadline_queue_init()` sudah berfungsi dan tidak perlu ditulis ulang. Lima fungsi berikut masih TODO. Admission API pada bagian 02 harus diputuskan lebih dahulu agar full - queue behavior benar - benar dapat dicapai.

### **Header contract: `firmware/endpoint/main/deadline_queue.h`**

~~~c
#ifndef CLDT_ENDPOINT_DEADLINE_QUEUE_H
#define CLDT_ENDPOINT_DEADLINE_QUEUE_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_ENDPOINT_MAX_POOL_SLOTS 128U

typedef enum {
    CLDT_SLOT_FREE = 0,
    CLDT_SLOT_PRODUCER_OWNED,
    CLDT_SLOT_QUEUED,
    CLDT_SLOT_TRANSPORT_OWNED
} cldt_slot_state_t;

typedef struct {
    /* Slot state is the ownership proof; no slot may have two owners at once. */
    cldt_slot_state_t state;
    cldt_frame_meta_t meta;
    uint16_t payload_bytes;
    uint8_t payload[CLDT_MAX_PAYLOAD_BYTES];
} cldt_message_slot_t;

typedef struct {
    /* order contains slot indices in deadline order; it is never a second payload store. */
    cldt_message_slot_t slots[CLDT_ENDPOINT_MAX_POOL_SLOTS];
    uint16_t order[CLDT_ENDPOINT_MAX_POOL_SLOTS];
    uint16_t capacity;
    uint16_t queued;
    uint16_t high_water;
    uint64_t rejected;
    uint64_t expired;
    uint64_t coalesced;
    uint64_t pool_exhaustions;
} cldt_deadline_queue_t;

/*
 * The queue has one task owner. ISRs notify the producer; they never call this
 * API. capacity must not exceed CLDT_ENDPOINT_MAX_POOL_SLOTS.
 */
cldt_status_t cldt_deadline_queue_init(
    cldt_deadline_queue_t *queue,
    uint16_t capacity);

/*
 * Transfers one free slot to the producer. The producer must fully initialize
 * metadata and payload before commit; it must release the slot on any local
 * generation failure instead of leaving a producer-owned leak.
 */
cldt_status_t cldt_deadline_queue_acquire(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t **slot);

/*
 * Transfers a completely initialized producer slot to the queue. The queue is
 * responsible for expiry and admission accounting after this call succeeds.
 */
cldt_status_t cldt_deadline_queue_commit(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t *slot,
    uint64_t now_local_us);

/*
 * Transfers the next live slot to the transport task. A successful pop does not
 * imply delivery; transport must produce a terminal trace then release the slot.
 */
cldt_status_t cldt_deadline_queue_pop(
    cldt_deadline_queue_t *queue,
    uint64_t now_local_us,
    cldt_message_slot_t **slot);

/* Returns a producer- or transport-owned slot to the pool after terminal trace. */
cldt_status_t cldt_deadline_queue_release(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t *slot);

/* Expires queued work and returns the number of slots released. */
cldt_status_t cldt_deadline_queue_expire(
    cldt_deadline_queue_t *queue,
    uint64_t now_local_us,
    uint16_t *expired_slots);

#ifdef __cplusplus
}
#endif

#endif

~~~

Source harus mempertahankan ownership state, fixed arrays, capacity, queue order, dan counter yang dideklarasikan di sini. Jika full - queue admission memerlukan perubahan API/struct, header dan seluruh caller diubah bersama; notebook tidak menamai field tambahan.

### **Source TODO asli**

~~~c
#include "deadline_queue.h"

#include <string.h>

cldt_status_t cldt_deadline_queue_init(
    cldt_deadline_queue_t *queue,
    uint16_t capacity)
{
    if (queue == NULL || capacity == 0 || capacity > CLDT_ENDPOINT_MAX_POOL_SLOTS) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    memset(queue, 0, sizeof(*queue));
    queue->capacity = capacity;

    return CLDT_OK;
}

cldt_status_t cldt_deadline_queue_acquire(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t **slot)
{
    if (queue == NULL || slot == NULL) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    // TODO: Acquire: linear scan slots[0..capacity-1] for CLDT_SLOT_FREE, transition to PRODUCER_OWNED
    // TODO: Capacity reservation: reserve N slots for CLDT_TRAFFIC_CONTROL and CLDT_TRAFFIC_CRITICAL classes

    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_deadline_queue_commit(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t *slot,
    uint64_t now_local_us)
{
    if (queue == NULL || slot == NULL) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    (void)now_local_us;

    // TODO: Commit: validate PRODUCER_OWNED state, check already-expired, binary search insert into order[], update high_water
    // TODO: The EXACT algorithm (EDF = Earliest Deadline First, sorted by deadline_local_us)
    // TODO: Binary search: compare deadline_local_us in order[] array, find insertion point, memmove to shift
    // TODO: Local EDF ordering is not an end-to-end schedulability proof. Retain
    // queue, transport, radio, and acknowledgement timing evidence.
    // TODO: Admission control: when queue is full, reject the item with the LATEST deadline (either incoming or last in queue)
    // TODO: Coalescing: for CLDT_TRAFFIC_TELEMETRY, if a queued item has same source node, replace the older one

    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_deadline_queue_pop(
    cldt_deadline_queue_t *queue,
    uint64_t now_local_us,
    cldt_message_slot_t **slot)
{
    if (queue == NULL || slot == NULL) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    (void)now_local_us;

    // TODO: Pop: always take order[0] (earliest deadline), shift array left, transition to TRANSPORT_OWNED

    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_deadline_queue_release(
    cldt_deadline_queue_t *queue,
    cldt_message_slot_t *slot)
{
    if (queue == NULL || slot == NULL) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    // TODO: Release: require PRODUCER_OWNED or TRANSPORT_OWNED, clear metadata bytes to prevent info leakage, transition to FREE

    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_deadline_queue_expire(
    cldt_deadline_queue_t *queue,
    uint64_t now_local_us,
    uint16_t *expired_slots)
{
    if (queue == NULL || expired_slots == NULL) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }

    (void)now_local_us;

    // TODO: Expiry sweep: the queue-owning task traverses order[] once, removes
    // expired items, and compacts survivors without changing relative order.
    // TODO: A 10 ms esp_timer callback only notifies that owner; it never walks or mutates the queue

    return CLDT_ERR_NOT_IMPLEMENTED;
}
~~~

### **Invariant yang tidak boleh putus**

| State | Pemilik sah | Transisi keluar |
| - | - | - |
| `CLDT_SLOT_FREE` | pool | `cldt_deadline_queue_acquire()` → producer - owned |
| `CLDT_SLOT_PRODUCER_OWNED` | producer task | `cldt_deadline_queue_commit()` atau `cldt_deadline_queue_release()` |
| `CLDT_SLOT_QUEUED` | queue owner | `cldt_deadline_queue_pop()`, expiry, coalesce, atau admission eviction |
| `CLDT_SLOT_TRANSPORT_OWNED` | terminal/local consumer pada build ini | terminal trace lalu `cldt_deadline_queue_release()` |

### **Urutan implementasi**

1. `cldt_deadline_queue_acquire()` memulai dengan `*slot = NULL` agar error tidak meninggalkan pointer lama. Slot dicari hanya dalam `[0, capacity)`.
2. Transition free → producer - owned terjadi satu kali. Bila tidak ada slot sesuai reservation rule, `pool_exhaustions` dan event authoritative diperbarui melalui owner yang benar.
3. `cldt_deadline_queue_commit()` memastikan pointer benar - benar menunjuk slot milik queue, state - nya producer - owned, payload length valid, traffic class valid, dan deadline absolute belum lewat.
4. Binary search mencari posisi pertama yang mempertahankan urutan deadline. Tie rule - misalnya sequence lebih kecil lebih dahulu - dibekukan agar deterministic.
5. `memmove` hanya memindahkan index pada `order[]`, bukan payload.
6. Saat telemetry coalescing dipakai, identity “same source” mengikuti keputusan bagian 02. Item lama mendapat terminal coalesced trace sebelum slot dilepas.
7. Full - queue admission membandingkan incoming dengan latest deadline hanya melalui API contract yang sudah mampu menyediakan slot incoming.
8. `cldt_deadline_queue_pop()` melakukan expiry lebih dahulu atau menolak item yang sudah terlambat. Success mengambil `order[0]`, menggeser index, mengurangi `queued`, dan memindahkan ownership.
9. `cldt_deadline_queue_release()` memverifikasi pointer range dan state producer/transport - owned, membersihkan metadata/payload, lalu mengembalikan state ke free.
10. `cldt_deadline_queue_expire()` dijalankan queue owner, bukan timer callback. Survivor dikompakkan tanpa mengubah relative order; setiap expired item menghasilkan terminal trace.
11. `high_water`, `rejected`, `expired`, `coalesced`, dan `pool_exhaustions` mempunyai satu lokasi update authoritative.

### **Case on - device yang dicatat**

| Case | Contoh format - ganti dengan hasil aktual |
| - | - |
| invalid capacity dan null argument | e.g. PASS - status sesuai contract |
| acquire sampai pool exhausted | e.g. PASS - slot berikutnya ditolak secara visible |
| producer - owned leak setelah generation failure | e.g. PASS - slot kembali `CLDT_SLOT_FREE` |
| insertion awal/tengah/akhir dan deadline tie | e.g. PASS - `order[]` deterministic |
| commit item yang sudah expired | e.g. PASS - terminal expiry tercatat |
| pop mempertahankan EDF order | e.g. PASS |
| telemetry coalescing | e.g. PASS - item lama mendapat terminal trace |
| reserved critical capacity | e.g. PASS - bulk tidak memakai slot cadangan |
| full - queue incoming lebih awal | e.g. BLOCKED sampai contract incoming - slot diselesaikan |
| full - queue incoming lebih lambat | e.g. BLOCKED sampai contract incoming - slot diselesaikan |
| expiry sweep dengan survivor order | e.g. PASS |
| release pointer asing/double release | e.g. PASS - keduanya ditolak |


## **Step 07: Absolute Release Intent, Minimal ISR, dan Bounded Stop**
**Track:** **`Endpoint - Workload Release`**

Source berikut ditempel utuh agar batasnya terlihat. Pada Week 2 yang diisi adalah `cldt_workload_init()`, `cldt_workload_start()`, `cldt_workload_event_isr()`, dan `cldt_workload_stop()`. `cldt_workload_apply_policy()` tetap mengembalikan `ESP_ERR_NOT_SUPPORTED` karena command/auth/policy baru masuk setelah shadow dan safety chain.

### **Header contract: `firmware/endpoint/main/workload.h`**

~~~c
#ifndef CLDT_ENDPOINT_WORKLOAD_H
#define CLDT_ENDPOINT_WORKLOAD_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include "freertos/timers.h"

#include "cldt/cldt_types.h"
#include "deadline_queue.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_ENDPOINT_MAX_STREAMS 8U

typedef struct {
    uint32_t stream_id;
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint32_t phase_ms;
    uint32_t jitter_ms;
    uint32_t deadline_ms;
    uint16_t payload_bytes;
    uint16_t burst_packets;
    uint16_t maximum_rate_pps;
} cldt_stream_config_t;

typedef struct {
    cldt_deadline_queue_t *queue;
    cldt_stream_config_t streams[CLDT_ENDPOINT_MAX_STREAMS];
    size_t stream_count;
    cldt_policy_t active_policy;
    TaskHandle_t producer_task;
    TimerHandle_t release_timer;
    uint32_t random_state;
    bool running;
} cldt_workload_t;

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed);

/* Starts release timers only after the run digest is accepted. */
esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us);

/* Applies a prevalidated immutable policy snapshot at a release boundary. */
esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy);

/* ISR entry: capture no payload and wake only the producer task. */
void cldt_workload_event_isr(void *context);

esp_err_t cldt_workload_stop(cldt_workload_t *workload);

#ifdef __cplusplus
}
#endif

#endif

~~~

Field `stream_id`, `traffic_class`, timing, payload, burst, `maximum_rate_pps`, `random_state`, dan `running` berasal dari header ini. Table workload hanya memakai nama tersebut.

### **Source TODO asli**

~~~c
#include "workload.h"

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed)
{
    (void)workload;
    (void)queue;
    (void)streams;
    (void)stream_count;
    (void)seed;

    /*
     * IMPLEMENTATION TODO: validate non-null arguments, stream count, unique
     * stream IDs, payload/deadline/rate bounds, and aggregate offered rate against
     * the endpoint safety limit. Copy the approved stream list into workload-owned
     * storage, seed a documented deterministic jitter generator, and create the
     * producer task and timer with static allocation. A failed init must leave
     * running false and must not alter the deadline queue.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us)
{
    (void)workload;
    (void)run_start_local_us;

    /*
     * IMPLEMENTATION TODO: require an accepted run start time and inactive
     * workload, calculate each first release from the same local monotonic epoch
     * plus its phase, and schedule absolute release intent rather than chaining
     * relative delays that accumulate jitter. Timer callbacks only notify the
     * producer task; payload creation, queue admission, tracing, and networking
     * happen in task context. Record release jitter against the intended time.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy)
{
    (void)workload;
    (void)policy;

    /*
     * IMPLEMENTATION TODO: accept only a policy already authenticated and checked
     * by endpoint runtime, copy it into a staging snapshot, and swap it at one
     * documented release boundary so no stream sees half old/half new fields.
     * Revalidate that critical periods and reserved queue capacity remain inside
     * compiled limits. Trace old epoch, new epoch, and effective local time; do
     * not dynamically allocate or edit the manifest at runtime.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

void cldt_workload_event_isr(void *context)
{
    (void)context;

    /*
     * IMPLEMENTATION TODO: keep this ISR to the minimum allowed by FreeRTOS:
     * validate the stored context if practical, call the appropriate FromISR task
     * notification primitive, capture whether a higher-priority task woke, and
     * request a yield through the documented port macro. Do not allocate, log,
     * acquire a mutex, encode a frame, or call OpenThread from this ISR.
     */
}

esp_err_t cldt_workload_stop(cldt_workload_t *workload)
{
    (void)workload;

    /*
     * IMPLEMENTATION TODO: stop or disarm release timers, signal producer task
     * to stop creating new work, wait a bounded time for transport-owned slots,
     * explicitly expire remaining queued work if the deadline passes, and report
     * a final accounting snapshot. Only then set running false. Preserve the
     * reason for forced expiry so a fast shutdown never becomes invisible loss.
     */
    return ESP_ERR_NOT_SUPPORTED;
}
~~~

### **Pekerjaan fungsi**

1. `cldt_workload_init()` memvalidasi stream count, stream ID unik, class, payload, period, phase, jitter, deadline, burst, dan rate. Approved configuration disalin; caller array tidak dipinjam.
2. Worst - case offered - rate bound untuk satu stream dengan full burst setiap period dapat dicatat sebagai:

$$
\text{offered rate per stream (packet/s)}
=
\frac{1000 \times \texttt{burst\_packets}}
{\texttt{period\_ms}}
$$

Jumlah offered rate seluruh stream harus berada di bawah Kconfig `CLDT_ENDPOINT_MAX_TOTAL_RATE_PPS`. Formula lain boleh dipakai hanya bila burst semantics - nya ditulis eksplisit.
3. Jitter PRNG deterministic memakai `seed` manifest. Algoritma dan update order dibekukan supaya run dapat direplay.
4. Static task/timer storage mengikuti keputusan bagian 02. Partial init dibersihkan dan `running` tetap false.
5. `cldt_workload_start()` menerima monotonic `run_start_local_us` nonzero dan menghitung first release dari epoch yang sama. Satu timer bangun pada nearest absolute release.
6. Timer callback hanya memberi notification. Producer task membuat payload, memperoleh slot, mengisi identity/timestamp/deadline, melakukan commit, dan mengirim trace.
7. `cldt_workload_event_isr()` hanya memakai FromISR notification, wake flag yang diwajibkan API FreeRTOS, dan yield macro yang sesuai. Nama local variable ditentukan saat implementasi; notebook tidak menambahkan identifier baru ke contract repo. Tidak ada log, allocation, mutex, encoding, atau OpenThread.
8. `cldt_workload_stop()` menutup release baru, menunggu ownership yang sah dengan batas waktu, mengekspirasi queued item yang tersisa, mengambil final metrics, lalu mengubah `running` menjadi false.
9. Forced expiry pada shutdown mempunyai reason dan terminal trace. Ia tidak dibuang sebagai “cleanup”.

### **Nilai workload yang dipilih setelah pilot**

| Stream | Field dari `cldt_stream_config_t` | Contoh awal - ganti setelah pilot |
| - | - | - |
| critical | `stream_id` | e.g. 1 |
| critical | `traffic_class` | e.g. `CLDT_TRAFFIC_CRITICAL` |
| critical | `period_ms` | e.g. 100 |
| critical | `phase_ms` | e.g. 0 |
| critical | `jitter_ms` | e.g. 5 |
| critical | `deadline_ms` | e.g. 80 |
| critical | `payload_bytes` | e.g. 32 |
| critical | `burst_packets` | e.g. 1 |
| critical | `maximum_rate_pps` | e.g. 20 |
| bulk | `stream_id` | e.g. 2 |
| bulk | `traffic_class` | e.g. `CLDT_TRAFFIC_BULK` |
| bulk | `period_ms` | e.g. 1000 |
| bulk | `phase_ms` | e.g. 0 |
| bulk | `jitter_ms` | e.g. 0 |
| bulk | `deadline_ms` | e.g. 900 |
| bulk | `payload_bytes` | e.g. 128 |
| bulk | `burst_packets` | e.g. 10 |
| bulk | `maximum_rate_pps` | e.g. 20 |
| semua stream | argumen `seed` pada `cldt_workload_init()` | e.g. 12345 |
| semua stream | jumlah offered rate hasil formula | e.g. 20 packet/s; harus ≤ Kconfig `CLDT_ENDPOINT_MAX_TOTAL_RATE_PPS` |


## **Step 08: Mengintegrasikan Komponen Tanpa Menyalakan Radio**
**Track:** **`Endpoint - Local Runtime`**

File runtime memuat lifecycle final yang lebih luas daripada Week 2. Snippet utuh dipertahankan agar fungsi deferred tidak tertukar dengan fungsi local. Pada fase ini:

- `cldt_endpoint_runtime_init()` mengalokasikan dan menginisialisasi state lokal;
- `cldt_endpoint_runtime_start()` menjalankan supervisor, trace, queue owner, dan workload local - only;
- `cldt_endpoint_runtime_request_stop()` menghasilkan shutdown serta reconciliation;
- `cldt_endpoint_runtime_receive_command()` tetap unsupported;
- transport dan power task tidak dibuat.

### **Header dan build contract yang terhubung**

#### `firmware/endpoint/main/endpoint_runtime.h`

~~~c
#ifndef CLDT_ENDPOINT_RUNTIME_H
#define CLDT_ENDPOINT_RUNTIME_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/event_groups.h"
#include "freertos/task.h"

#include "cldt/cldt_clock_sync.h"
#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_types.h"
#include "deadline_queue.h"
#include "power_probe.h"
#include "workload.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_ENDPOINT_BOOT = 0,
    CLDT_ENDPOINT_COMMISSIONING,
    CLDT_ENDPOINT_ATTACHED,
    CLDT_ENDPOINT_IDLE,
    CLDT_ENDPOINT_RUNNING,
    CLDT_ENDPOINT_FALLBACK,
    CLDT_ENDPOINT_FAULT
} cldt_endpoint_state_t;

typedef struct {
    cldt_node_id_t node_id;
    cldt_node_role_t role;
    cldt_endpoint_state_t state;
    /* RAM mirrors loaded from an integrity-checked durable replay record. */
    cldt_run_id_t active_run_id;
    cldt_boot_id_t command_authority_boot_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t next_sequence;
    cldt_policy_epoch_t applied_epoch;
    cldt_policy_t safe_policy;
    cldt_policy_t active_policy;
    cldt_clock_sync_t clock_sync;
    cldt_deadline_queue_t deadline_queue;
    cldt_workload_t workload;
    cldt_event_trace_t trace;
    EventGroupHandle_t events;
    TaskHandle_t supervisor_task;
    TaskHandle_t transmitter_task;
    TaskHandle_t trace_task;
    TaskHandle_t power_task;
    /* False forbids remote apply and keeps the compiled safe policy active. */
    bool replay_state_valid;
    bool started;
} cldt_endpoint_runtime_t;

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime);
esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime);

/* Validates coordinator identity, run, durable epoch, TTL, and local limits. */
esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us);

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime);

#ifdef __cplusplus
}
#endif

#endif

~~~

#### `firmware/endpoint/main/CMakeLists.txt`

~~~cmake
idf_component_register(
    SRCS
        "app_main.c"
        "endpoint_runtime.c"
        "deadline_queue.c"
        "workload.c"
        "thread_transport.c"
        "power_probe.c"
    INCLUDE_DIRS "."
    REQUIRES
        common
        freertos
        nvs_flash
        esp_event
        esp_netif
        esp_timer
        driver
        openthread
)

~~~

#### `firmware/endpoint/main/Kconfig.projbuild`

~~~text
menu "CLDT Endpoint"

config CLDT_ENDPOINT_NODE_ID
    int "Provisioning fallback node ID"
    range 1 4294967295
    default 100
    help
        Used only before the endpoint receives a provisioned identity. Record the
        final identity in run evidence; do not use a MAC address or an ad hoc hash.

choice CLDT_ENDPOINT_ROLE
    prompt "Endpoint role"
    default CLDT_ENDPOINT_ROUTER

config CLDT_ENDPOINT_ROUTER
    bool "Router-capable endpoint"
    help
        Enables the project role expected to be router-capable. Actual Thread role
        remains an observed run fact and must not be inferred from this selection.

config CLDT_ENDPOINT_LOW_POWER
    bool "Low-power end-device candidate"
    help
        Selects a candidate end-device configuration for a later admitted study.
        Actual Thread role remains observed, and no version-one energy claim is
        implied by this build choice.

endchoice

config CLDT_ENDPOINT_POOL_SLOTS
    int "Fixed message slots"
    range 8 128
    default 32
    help
        Compile-time queue-pool capacity. Choose from measured peak occupancy and
        preserve a control/critical reservation; never grow it only to hide loss.

config CLDT_ENDPOINT_MAX_TOTAL_RATE_PPS
    int "Compiled maximum aggregate application rate"
    range 1 500
    default 100
    help
        Hard local ceiling across all application streams. A ready manifest may
        request less, but no host policy can raise this value during a run.

config CLDT_ENDPOINT_EVENT_GPIO
    int "Local event input GPIO"
    range -1 30
    default -1
    help
        -1 disables the optional physical event input. Select a pin only after
        checking the exact board revision, boot strapping, pull mode, and debounce
        behavior; XIAO GPIO9 is a boot input and is not a safe generic default.

config CLDT_ENDPOINT_I2C_SDA_GPIO
    int "Optional power-probe I2C SDA GPIO"
    range 0 30
    default 22
    help
        XIAO ESP32-C6 D4/SDA is GPIO22. The version-one power probe is deferred;
        recheck the exact board and record wiring before a future energy pilot.

config CLDT_ENDPOINT_I2C_SCL_GPIO
    int "Optional power-probe I2C SCL GPIO"
    range 0 30
    default 23
    help
        XIAO ESP32-C6 D5/SCL is GPIO23. Do not assume another C6 board variant
        shares this mapping.

endmenu

~~~

Header memperlihatkan state dan handle yang benar - benar tersedia. CMake memperlihatkan bahwa `thread_transport.c` dan `power_probe.c` ikut terkompilasi walaupun jalurnya belum dipanggil pada local - only build; Kconfig memberi batas pool/rate dan pin yang tidak boleh diganti oleh nama konfigurasi buatan notebook.

#### `firmware/endpoint/CMakeLists.txt`

Root firmware file memilih ESP - IDF project dan common component; ia tidak membuktikan target telah dikonfigurasi.

~~~cmake
cmake_minimum_required(VERSION 3.16)

set(EXTRA_COMPONENT_DIRS "${CMAKE_CURRENT_LIST_DIR}/../../common")

include($ENV{IDF_PATH}/tools/cmake/project.cmake)
project(cldt_endpoint)
~~~

#### `firmware/endpoint/sdkconfig.defaults`

Defaults berikut menyalakan OpenThread dan crypto/trace support. Untuk local - only pilot, status radio tetap harus dibuktikan dari runtime path; defaults ini sendiri tidak berarti Thread aktif maupun nonaktif saat run.

~~~ini
CONFIG_FREERTOS_HZ=1000
CONFIG_FREERTOS_USE_TRACE_FACILITY=y
CONFIG_FREERTOS_GENERATE_RUN_TIME_STATS=y
CONFIG_ESP_TASK_WDT_EN=y
CONFIG_OPENTHREAD_ENABLED=y
CONFIG_MBEDTLS_CHACHAPOLY_C=y
CONFIG_MBEDTLS_CHACHA20_C=y
CONFIG_MBEDTLS_POLY1305_C=y
~~~


### **Source TODO asli**

~~~c
#include "endpoint_runtime.h"

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: reject a null runtime, clear caller-owned state, load
     * immutable board identity and role, generate a boot ID that changes after a
     * reset, and load the integrity-checked durable replay record containing the
     * enrolled run, coordinator boot identity, and highest accepted epoch. A
     * missing or corrupt record leaves
     * replay_state_valid false: retain the compiled safe policy and require an
     * explicitly commissioned new unique run before remote apply. Create every
     * steady-state queue, trace buffer, event group, and task storage statically
     * and leave the state at BOOT. No radio attach, workload release, or dynamic
     * allocation is permitted here. Fail before changing externally visible
     * state on any error.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: require successful initialization, then start the
     * supervisor task first. It owns transitions through commissioning, attach,
     * idle, running, fallback, and fault. Start transport, workload, trace, and
     * optional power tasks only after the supervisor reports their prerequisites;
     * if any task creation fails, notify supervisor to unwind already started
     * components. Do not start release timers merely because Thread attached.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us)
{
    (void)runtime;
    (void)datagram;
    (void)datagram_bytes;
    (void)received_local_us;

    /*
     * IMPLEMENTATION TODO:
     * 1. Copy or retain the datagram only for the duration required by the
     *    decoder; reject oversized input before queueing work.
     * 2. Decode and authenticate it; require coordinator authority node ID 0 and
     *    the coordinator boot/session identity commissioned for this run, the
     *    enrolled run ID, a valid durable replay state, a strictly newer epoch,
     *    a live TTL, and endpoint-local limits. Map received_local_us into the
     *    gateway monotonic domain through the validated clock-sync state and
     *    reject excessive uncertainty; never compare unrelated local clocks.
     *    The command boot ID identifies the coordinator process, not this
     *    endpoint and not replay state. A host decision is not local authorization.
     * 3. Atomically persist the new (run_id, coordinator_boot_id, highest_epoch)
     *    before publishing
     *    one immutable policy snapshot at a workload release boundary. If the
     *    durable write fails, reject and retain the safe policy. A duplicate
     *    accepted epoch must be acknowledged as duplicate, never applied twice.
     * 4. Emit a trace record and an acknowledgement for every accept or reject
     *    reason. On missing/corrupt replay state or any ambiguity, preserve the
     *    safe policy, enter FALLBACK, and require a newly commissioned run.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: request a supervisor-owned stop, block new workload
     * releases, let producer and transport finish or explicitly expire queued
     * work, request final counters, and reconcile before changing state to IDLE.
     * A stopped endpoint must retain its safe policy and remain able to report
     * health. Do not delete a task from an arbitrary caller or discard evidence
     * merely to make shutdown appear fast.
     */
    return ESP_ERR_NOT_SUPPORTED;
}
~~~

### **Integrasi local - only**

1. `cldt_endpoint_runtime_init()` menolak null, membersihkan state, memuat `node_id`/role, membuat `boot_id` baru, menginisialisasi queue, metrics, trace storage, event group, dan static task storage.
2. Remote replay state tidak diperlukan untuk local pilot, tetapi `replay_state_valid` tetap false dan safe policy tetap aktif. Jangan membuat replay record palsu.
3. State transition local - only harus ditulis eksplisit. Ia tidak boleh mengklaim `CLDT_ENDPOINT_ATTACHED` ketika OpenThread tidak pernah dimulai.
4. `cldt_endpoint_runtime_start()` memulai supervisor dahulu. Supervisor mengakui prerequisites lokal sebelum workload timer dimulai.
5. Transport, command callback, Wi - Fi, OpenThread, dan power probe tidak dipanggil pada binary ini.
6. Terminal local consumer menghasilkan terminal trace untuk setiap popped slot, lalu melepaskan ownership. Tidak ada fabricated UDP ACK.
7. `cldt_endpoint_runtime_request_stop()` menghentikan release, menyelesaikan/mengekspirasi item, men - drain trace, mengambil final counters, menjalankan aggregate reconciliation serta item audit, lalu masuk state akhir.
8. `app_main()` tetap tipis: object runtime, init, start, dan penyerahan lifecycle ke supervisor. Pilot scenario bukan hard - coded loop besar di entry point.

### **Entry point saat ini**

~~~c
#include "esp_log.h"

static const char *TAG = "cldt_endpoint";

void app_main(void)
{
    ESP_LOGW(TAG,
             "Research scaffold only: endpoint runtime, deadline queue, "
             "workload, Thread transport, and power probe are not implemented.");

    /*
     * IMPLEMENTATION ORDER:
     * 1. Keep networking disabled while proving local software-timer, ISR,
     *    queue ownership, fixed-pool exhaustion, expiry, and counter tests.
     * 2. Add Thread attachment only after those tests produce reconciled traces.
     * 3. Add command handling only after protocol known-answer and replay tests.
     * 4. Enable the optional power probe last and document its overhead.
     * This entry point should remain small: construct runtime, call init/start,
     * and hand lifecycle ownership to the supervisor. It is not a demo script.
     */
}
~~~

### **Bukti bahwa radio tidak aktif**

| Bukti local - only | Contoh format - ganti dengan hasil aktual |
| - | - |
| Jalur build/start yang dipakai | e.g. catatan perubahan di `cldt_endpoint_runtime_start()` |
| Source commit | e.g. 40 karakter hexadecimal |
| SHA - 256 `sdkconfig` endpoint | e.g. 64 karakter hexadecimal |
| SHA - 256 binary endpoint | e.g. 64 karakter hexadecimal |
| Lokasi cold - boot log | e.g. C:\logs\cldt\endpoint - local - boot.txt |
| Pemanggilan init OpenThread | e.g. tidak ditemukan pada jalur local - only |
| Pemanggilan init Wi - Fi | e.g. tidak ditemukan pada jalur local - only |
| Urutan nilai `cldt_endpoint_runtime_t.state` | e.g. `CLDT_ENDPOINT_BOOT` → `CLDT_ENDPOINT_IDLE` → `CLDT_ENDPOINT_RUNNING` → `CLDT_ENDPOINT_IDLE` |
| Perubahan `cldt_endpoint_runtime_t.boot_id` setelah reset | e.g. PASS - dua boot menghasilkan nilai berbeda |


## **Step 09: Memilih Parameter dari Trace, Bukan dari Angka Cantik**
**Track:** **`Physical Endpoint - Three Pilots`**

Pilot dilakukan pada satu C6 dengan Thread dan Wi - Fi tidak diinisialisasi. Ketiganya non - reportable dan menggunakan source/build identity yang sama. Satu faktor diubah pada tiap langkah agar asal queue pressure dapat dibaca.

### **Pilot 1 - critical periodic saja**

1. Satu stream critical dijalankan dengan burst satu.
2. Period, deadline, dan duration dipilih agar banyak release terlihat tanpa overlap yang belum dipahami.
3. Trace minimum: release, acquire, enqueue, dequeue/local start, local finish/terminal, release slot.
4. Hasil yang dicari: timer jitter, normal queue depth, terminal local path, dan conservation tanpa pressure.

### **Pilot 2 - bounded bulk burst saja**

1. Critical stream dimatikan; satu bulk burst digunakan.
2. Burst dinaikkan sampai admission, pool exhaustion, rejection, dan expiry dapat diuji secara terkendali.
3. Capacity tidak dibesarkan selama pilot yang sama.
4. Hasil yang dicari: high - water, full - pool rule, latest - deadline admission, terminal reject/expire, dan tidak ada slot leak.

### **Pilot 3 - critical dan bulk bersama**

1. Nilai dari dua pilot sebelumnya digabung tanpa mengubah algorithm.
2. Reserved critical capacity harus terlihat bekerja saat bulk memberi pressure.
3. Critical deadline outcome serta bulk rejection/expiry dicatat bersamaan.
4. Hasil yang dicari: semua lifecycle reconcile dan critical path tidak hilang diam - diam.

Jumlah release rencana untuk stream periodik pada measured window:

$$
\text{planned releases}
=
\left\lfloor
\frac{1000 \times \texttt{measurement\_s}}
{\texttt{period\_ms}}
\right\rfloor
\times \texttt{burst\_packets}
$$

Actual release tetap dihitung dari trace karena phase, jitter, warm - up, cooldown, dan boundary dapat mengubah jumlah. Planned count adalah cross - check, bukan pengganti raw event.

### **Rekaman pilot**

| Pilot | Seed | Duration | Released | Local terminal | Expired | Coalesced | Rejected | Dropped | Unresolved | High - water | Pool exhaustion | Reconciled |
| - :| - :| - :| - :| - :| - :| - :| - :| - :| - :| - :| - :| - |
| critical | e.g. 12345 | e.g. 120 s | e.g. 1200 | e.g. 1200 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. 2 | e.g. 0 | e.g. PASS |
| bulk | e.g. 12345 | e.g. 120 s | e.g. 1200 | e.g. 1050 | e.g. 50 | e.g. 0 | e.g. 100 | e.g. 0 | e.g. 0 | e.g. 32 | e.g. 0 | e.g. PASS |
| combined | e.g. 12345 | e.g. 120 s | e.g. 2400 | e.g. 2200 | e.g. 50 | e.g. 0 | e.g. 150 | e.g. 0 | e.g. 0 | e.g. 32 | e.g. 0 | e.g. PASS |

### **Evidence per pilot**

- [ ] source commit, `sdkconfig`, dan binary SHA - 256;
- [ ] boot ID dan local run identity;
- [ ] raw trace utuh;
- [ ] final queue counters;
- [ ] aggregate reconciliation;
- [ ] item - level audit;
- [ ] serial boot/stop log;
- [ ] operator note untuk forced expiry, reset, overflow, atau manual interruption.

Nilai manifest baru dipilih setelah ketiga row terisi. Pilot yang gagal tetap disimpan dan tidak dimasukkan sebagai repetisi final.


## **Step 10: Mengisi JSONC dari Bukti, Lalu Menilai Kelayakan Strict JSON**
**Track:** **`Manifest - Local RTOS Baseline`**

Planning dimulai pada `experiments/authoring/local - rtos - baseline.jsonc` karena di sana komentar field tersedia. Completed value kemudian disalin ke `experiments/local - rtos - baseline.json`. Strict file berikut masih template dan seluruh `null` memang belum boleh ditebak.

### **Planning companion asli: `experiments/authoring/local - rtos - baseline.jsonc`**

~~~jsonc
{
  // This is the authoring guide for ../local-rtos-baseline.json.
  // Thread and Wi-Fi must remain inactive for this local-accounting experiment.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "local-rtos-accounting",
  "title": "Local RTOS Accounting Baseline",
  "purpose": {
    "question": "Can one endpoint account for every released item before Thread networking is introduced?",
    "comparison": "Critical and bulk queue behavior on a local endpoint with networking disabled.",
    "primary_metric": "Zero unreconciled items across release, queue, expiry, and terminal accounting."
  },
  "setup": {
    // Use the actual endpoint label; retain later topology roles only as context, not as a dependency.
    "nodes": null,
    // Record the intended later Thread channel only if it is known; radio attachment remains disabled here.
    "thread_channel": null,
    // State bench position, power source, and that no Thread network is active.
    "placement": null,
    // Include endpoint source revision, sdkconfig hash, binary hash, and local-test build option.
    "firmware_reference": null
  },
  "execution": {
    // Warm-up allows timers and trace buffers to reach steady state before measurement.
    "warmup_s": null,
    // Include enough releases to exercise periodic critical work and bounded bulk burst behavior.
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    // The seed must reproduce workload release and any deterministic burst pattern.
    "seed": null
  },
  "traffic": {
    // Define one critical and one bulk stream with explicit period, deadline, payload, and burst.
    "streams": null
  },
  "scenario": {
    // Use event none unless one declared local queue disturbance is introduced.
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Local accounting is baseline behavior: no model and no remote policy.
    "mode": null,
    "candidate_action": null,
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Set true only after the implementation can reconcile each logical work item.
    "counter_reconciliation": null,
    // Choose a local service floor only if clock/deadline accounting is ready to support it.
    "minimum_critical_on_time_pdr": null,
    // Require an explicit invalidation rule such as missing terminal trace records.
    "negative_case": null
  },
  "evidence": {
    // Require raw local trace, counters, binary identity, manifest, and operator notes.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Create a reproducible local endpoint build.",
      "method": "Archive board label, firmware identity, power path, and proof that Thread attachment is disabled.",
      "done_when": "Cold boot reaches local test mode without radio attachment."
    },
    {
      "path": "/execution",
      "action": "Select timing from traceable pilots.",
      "method": "Exercise critical-only, bulk-only, and combined paths; capture release, admission, expiry, and terminal events.",
      "done_when": "Every intended path appears and no released item is unexplained."
    },
    {
      "path": "/acceptance",
      "action": "Turn the result into a real accounting gate.",
      "method": "Require reconciliation and retain evidence that lets another reader recompute terminal outcomes offline.",
      "done_when": "No network performance claim is needed to validate the local result."
    }
  ]
}
~~~

### **Strict template asli**

~~~json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "local-rtos-accounting",
  "title": "Local RTOS Accounting Baseline",
  "purpose": {
    "question": "Can one endpoint account for every released item before Thread networking is introduced?",
    "comparison": "Critical and bulk queue behavior on a local endpoint with networking disabled.",
    "primary_metric": "Zero unreconciled items across release, queue, expiry, and terminal accounting."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Define the one endpoint that will run the local queue test, while explicitly keeping Thread and Wi-Fi inactive.",
      "method": "Record the actual C6 board label, ESP-IDF revision, sdkconfig hash, binary hash, power source, and bench position. Set the four physical roles only if the later Thread topology is already assembled; this local run must not depend on that network.",
      "done_when": "A reproducible firmware reference exists and a cold boot reaches the local test mode without attempting radio attachment."
    },
    {
      "path": "/execution",
      "action": "Choose timing, seed, and two local streams from queue capacity and pilot traces rather than arbitrary values.",
      "method": "Run three non-reportable pilots: first a critical periodic stream, then a bounded bulk burst, then their combination. Capture timer release, queue admission, expiry, and terminal events; set duration and repetition count only after every intended path appears and every released item reconciles.",
      "done_when": "The ready file has a concrete duration, seed plan, and stream values supported by pilot traces with no unexplained counter mismatch."
    },
    {
      "path": "/acceptance",
      "action": "Turn the local result into a valid accounting gate rather than a vague smoke test.",
      "method": "Require counter reconciliation, define whether valid local timing permits a critical on-time floor, list the minimal evidence files, and explain any forced expiry or rejection in operator notes. Do not claim network latency in this scenario.",
      "done_when": "The evidence bundle lets another reader recompute each terminal outcome without a broker, dashboard, or host model."
    }
  ]
}
~~~

### **Sumber setiap field**

| Field repo | Aturan schema / design | Sumber nilai | Contoh bentuk nilai - bukan hasil |
| - | - | - | - |
| `setup.nodes` | ready membutuhkan tepat 4 node unik | label gateway, RCP, endpoint A, endpoint B | e.g. empat object dengan `id`, `board`, dan `role` yang cocok dengan label fisik |
| `setup.thread_channel` | integer 11-26 walau radio local run mati | channel fisik yang sudah dibekukan pada Day 10 | e.g. 15 |
| `setup.placement` | teks konkret | posisi endpoint, power source, cable, dan pernyataan Thread/Wi - Fi tidak aktif | e.g. endpoint A di meja utara, USB hub powered, radio tidak diinisialisasi |
| `setup.firmware_reference` | teks konkret | source commit, ESP - IDF commit, build path, `sdkconfig` hash, binary hash | e.g. satu string yang memuat kelima identity tersebut |
| `execution.warmup_s` | 5-300 | pilot: waktu sampai timer/trace steady | e.g. 30 |
| `execution.measurement_s` | 30-3600 | cukup untuk jumlah release dan path yang diperlukan | e.g. 300 |
| `execution.cooldown_s` | 5-300 | cukup untuk stop, final snapshot, dan trace drain | e.g. 30 |
| `execution.repetitions` | 1-30 | jumlah dibekukan sebelum final run | e.g. 5 |
| `execution.seed` | 0-4294967295 | seed deterministic yang direplay pada workload | e.g. 12345 |
| `traffic.streams` | 1-4 stream | stream dari tiga pilot | e.g. satu object critical dan satu object bulk |
| `period_ms` | 10-3.600.000 | interval release yang diuji | e.g. 100 |
| `payload_bytes` | 1-256 | payload local yang benar - benar dibuat | e.g. 32 |
| `deadline_ms` | 10-3.600.000 | deadline relative terhadap release | e.g. 80 |
| `burst_packets` | 1-100 | burst yang lulus capacity/rate check | e.g. 1 |
| `scenario.event` | enum schema | local baseline tidak mempunyai disturbance | e.g. `none` |
| `scenario.at_s` / `scenario.duration_s` | integer ≥0 | 0 untuk `none` | e.g. 0 / 0 |
| `scenario.target` | non - empty | deskripsi kondisi | e.g. no injected disturbance |
| `treatment.mode` | enum schema | tujuan manifest | e.g. `baseline` |
| `treatment.candidate_action` | enum schema | tidak ada policy | e.g. `none` |
| `treatment.control_profile` | ready wajib non - empty | registry profile yang resolved dan diarsipkan | e.g. UNRESOLVED - tetap template sampai profile sungguhan ada |
| `treatment.host_model` | boolean | local baseline | e.g. false |
| `treatment.remote_actuation` | boolean | local baseline | e.g. false |
| `acceptance.counter_reconciliation` | ready harus true | aggregate dan item audit | e.g. true hanya setelah keduanya lulus |
| `acceptance.minimum_critical_on_time_pdr` | 0-1 | arti local terminal/on - time harus selesai | e.g. UNRESOLVED sampai contract local terminal selesai |
| `acceptance.negative_case` | non - empty | failure yang diuji | e.g. missing terminal invalidates the run |
| `evidence.required_artifacts` | 4-8 item unik | artifact yang benar - benar diproduksi | e.g. manifest.json, versions.json, events.ndjson, final counters, operator notes |
| `evidence.operator_notes_required` | ready harus true | operator note wajib | e.g. true |
| `evidence.topology_photo_required` | boolean | keputusan evidence untuk bench lokal | e.g. false |

### **Stream object yang akan diisi**

~~~json
{
  "id": null,
  "source": null,
  "class": null,
  "period_ms": null,
  "payload_bytes": null,
  "deadline_ms": null,
  "burst_packets": null
}
~~~

Snippet di atas adalah bentuk kerja, bukan JSON yang sudah valid terhadap branch `ready`. Setiap `null` diganti hanya dari row pilot dan identity board yang nyata.


## **Step 11: Kapan Setiap `_todo` Boleh Dihapus**
**Track:** **`Manifest - Three TODO Objects`**

Strict manifest memuat tiga TODO object: `/setup`, `/execution`, dan `/acceptance`. Mereka tidak dihapus satu per satu hanya karena sebagian field terisi.

### **`/setup` selesai ketika**

1. Endpoint fisik yang dipakai mempunyai label, revision, port, power source, dan bench position.
2. Source commit, ESP - IDF commit, local - only activation method, `sdkconfig` hash, serta binary hash tersedia.
3. Cold boot mencapai local test mode tanpa OpenThread/Wi - Fi initialization.
4. Empat physical role dapat ditulis tanpa mengarang board. Schema branch `ready` memang meminta tepat empat node.
5. Channel intended untuk topologi fisik sudah dibekukan, walaupun radio local test tetap mati.

Jika empat node atau channel belum tersedia, JSONC boleh memuat catatan kerja, tetapi strict manifest tetap `state: "template"` dengan TODO setup.

### **`/execution` selesai ketika**

1. Critical - only, bulk - only, dan combined pilot telah dijalankan.
2. Period, deadline, payload, burst, phase/jitter rule, dan seed berasal dari trace.
3. Warm - up, measurement, cooldown, serta repetitions menghasilkan cukup release dan seluruh path penting.
4. Worst - case offered rate berada di bawah compiled limit.
5. Queue capacity dipilih dari measured occupancy dan reservation rule, bukan untuk menyembunyikan rejection.
6. Final parameters dibekukan sebelum repetisi reportable.

### **`/acceptance` selesai ketika**

1. Local terminal event dan on - time rule sudah memiliki contract yang tidak memakai fake ACK.
2. Aggregate reconciliation dan item audit dapat dihitung ulang dari raw trace.
3. `counter_reconciliation` dapat diisi true berdasarkan implementasi, bukan niat.
4. `minimum_critical_on_time_pdr` memiliki arti yang dapat dipertahankan. Bila field itu masih bernama network PDR sementara run tidak memiliki network, strict manifest tidak dipromosikan sampai semantic contract dibenahi.
5. Minimal empat artifact benar - benar diproduksi - misalnya manifest, versions/build identity, raw local trace, final counters/reconciliation, dan operator notes.
6. `negative_case` ditentukan sebelum final repetitions.

### **Dua blocker branch `ready`**

- Ready manifest memerlukan non - empty `control_profile`, tetapi local run tidak memakai policy. Nama profile hanya boleh diisi bila immutable baseline profile serta identity/digest - nya benar - benar ada.
- Ready manifest memerlukan empat node dan channel walaupun pertanyaan eksperimennya satu endpoint tanpa radio.

Bila blocker itu belum ditutup secara legitimate, hasil Week 2 tetap merupakan pilot engineering yang lengkap, sedangkan manifest tetap template. Mengosongkan `_todo` atau mengganti `state` hanya agar validator hijau tidak diperbolehkan.

### **Candidate artifact list**

| Artifact yang disebut contract repo | Contoh path - ganti dengan file aktual |
| - | - |
| Exact strict manifest bytes | e.g. results\local - pilot - 01\manifest.json |
| Versions/build identity | e.g. results\local - pilot - 01\versions.json |
| Raw local trace | e.g. results\local - pilot - 01\events.ndjson |
| Final counters | e.g. results\local - pilot - 01\final - counters.json |
| Aggregate reconciliation | e.g. results\local - pilot - 01\reconciliation.json |
| Item audit | e.g. results\local - pilot - 01\item - audit.json |
| Serial/operator log | e.g. results\local - pilot - 01\operator - notes.txt |
| Terminal status | e.g. results\local - pilot - 01\run - status.json |


## **Step 12: Memperluas Topologi Setelah Local dan Day - 10 Gate Jelas**
**Track:** **`Hardware - Endpoint B`**

Attachment endpoint B memakai image upstream `ot_cli`, bukan project `thread_transport.c`. Endpoint A dikembalikan ke image upstream Day 10 dengan hash/configuration yang sama; endpoint B memakai build yang cocok. Tujuannya hanya membuktikan topologi fisik dua endpoint yang stabil sebelum frame project dan recorder Week 3.

### **Checklist meja hardware**

- [ ] Endpoint B adalah XIAO ESP32 - C6 yang revision - nya dicatat.
- [ ] Kabel data endpoint B telah diuji dan diberi label.
- [ ] Powered hub dengan adapter mampu menjalankan S3, RCP, endpoint A, dan endpoint B bersamaan.
- [ ] Tidak ada board yang mendapat dua sumber VBUS/rail tanpa isolation design.
- [ ] Image `ot_cli`, ESP - IDF commit, target, dan `sdkconfig` cocok antara endpoint A dan B.
- [ ] Binary hash kedua endpoint dicatat; equality tidak diasumsikan.
- [ ] Board position, orientation, cable routing, dan power path difoto.
- [ ] Endpoint B menerima dataset yang sama melalui jalur privat.
- [ ] Actual role, parent, RLOC16, partition, IPv6, dan attach time dicatat untuk kedua endpoint.
- [ ] Ping/UDP reachability diperiksa tanpa frame CLDT, host model, atau actuation.
- [ ] Brownout, reconnect, detach, atau role change tetap masuk log.
- [ ] Rp 175.000 reserve tidak berubah menjadi budget sensor/aksesori.

### **Rekaman dua endpoint**

| Catatan fisik | Endpoint A - contoh format | Endpoint B - contoh format |
| - | - | - |
| Board/revision | e.g. XIAO ESP32 - C6; revision dari PCB | e.g. XIAO ESP32 - C6; revision dari PCB |
| Port serial | e.g. COM9 | e.g. COM10 |
| SHA - 256 binary | e.g. 64 karakter hexadecimal | e.g. 64 karakter hexadecimal |
| SHA - 256 `sdkconfig` | e.g. 64 karakter hexadecimal | e.g. 64 karakter hexadecimal |
| Actual Thread role | e.g. router | e.g. child |
| Parent RLOC16 | e.g. 0x1400 atau not applicable untuk router | e.g. 0x1400 |
| Own RLOC16 | e.g. 0x4000 | e.g. 0x4401 |
| Partition ID | e.g. 123456789 | e.g. 123456789 |
| Alamat IPv6 yang diuji | e.g. fdxx:xxxx:xxxx:xxxx::a | e.g. fdxx:xxxx:xxxx:xxxx::b |
| Attachment time | e.g. 12.1 s | e.g. 14.8 s |
| Ping | e.g. PASS - 5/5 replies | e.g. PASS - 5/5 replies |
| UDP | e.g. PASS - sequence diterima | e.g. PASS - sequence diterima |
| Reset tak terduga | e.g. 0 | e.g. 0 |
| Detach | e.g. 0 | e.g. 0 |

Data setup tersebut mulai mengisi planning copy `baseline.jsonc`. Execution dan acceptance stable Thread baseline tetap belum dibekukan sampai project frames, recorder, dan reconciliation Week 3 tersedia. Label “low - power endpoint” juga belum menjadi bukti actual sleepy/end - device role atau energy saving.


## **Urutan 10-14: Menutup Week 2 Tanpa Membawa Utang Tak Terlihat**

| No. | Fokus utama | Output akhir |
| - :| - | - |
| 10 | verifikasi verdict gate; bekukan local - only, static storage, admission API, timer scheduler, dan terminal lokal | contract implementasi yang tidak ambigu |
| 11 | isi event - trace ring dan metrics reducer/audit; mulai test host | trace tidak overwrite; mapping accounting eksplisit |
| 12 | isi EDF queue serta workload init/start/ISR/stop | ownership, expiry, reservation, dan absolute release dapat diuji |
| 13 | integrasikan runtime local - only; jalankan critical, bulk, dan combined pilot | raw trace serta reconciliation untuk memilih manifest values |
| 14 | bekukan planning manifest, jalankan final local block bila eligible, lalu attach endpoint B dengan image upstream | local accounting verdict dan topologi dua endpoint terdokumentasi |

Code dan hardware dapat berjalan paralel, tetapi dependency tetap sama. Endpoint B boleh disiapkan/di - flash lebih awal; ia tidak bergabung sebelum entry gate PASS. Manifest authoring boleh dibuka sejak awal; nilai execution dan acceptance tidak dibekukan sebelum pilot.

Tidak ada file `.py` pada jalur implementasi Week 2. File manifest yang dikerjakan hanya `experiments/authoring/local - rtos - baseline.jsonc` dan pasangan strict `experiments/local - rtos - baseline.json`; tujuh pasangan eksperimen lain tidak ikut diisi pada fase ini.

### **Closure Week 2**

- [ ] Day - 10 one - endpoint gate masih PASS pada configuration yang dipakai.
- [ ] Local project firmware cold boot tanpa OpenThread/Wi - Fi initialization.
- [ ] Event trace menolak overflow secara visible dan tidak overwrite evidence lama.
- [ ] Queue ownership tidak leak atau double - own.
- [ ] Critical reservation, expiry, rejection, coalescing, dan pool exhaustion diuji.
- [ ] Workload memakai absolute release intent dan deterministic seed.
- [ ] Setiap local work item memiliki identity serta satu terminal atau explicit unresolved status.
- [ ] Aggregate reconciliation dan item audit sama - sama konsisten.
- [ ] Tiga pilot tersimpan, termasuk yang gagal.
- [ ] JSONC berisi hanya nilai yang sudah dibuktikan.
- [ ] Strict JSON tetap template bila empat - node/channel/profile/semantic blockers belum selesai.
- [ ] Endpoint B attached setelah gate, dengan actual topology state dan raw log.
- [ ] Project protocol, command path, power probe, recorder, serta model tetap belum aktif.



> **Stop rule akhir Week 2:** bila lifecycle lokal belum reconcile, Week 3 tidak dimulai dari model atau dashboard. Pekerjaan berikutnya tetap pada evidence chain - project frame, bounded bridge, recorder, dan reconciliation - setelah accounting defect ditutup.
